# Fintech Loan Classification with DistilBERT
End-to-end notebook for training a sequence classification model.

In [ ]:
!pip install transformers datasets evaluate scikit-learn -q

In [ ]:
from datasets import Dataset

data = [
    {"text": "My payment failed but money deducted", "label": "payment"},
    {"text": "App crashes on login", "label": "tech"},
    {"text": "Need refund for last transaction", "label": "refund"},
    {"text": "Unable to complete payment", "label": "payment"},
    {"text": "Screen freezes after update", "label": "tech"},
    {"text": "Refund not processed", "label": "refund"}
]

dataset = Dataset.from_list(data)
dataset

In [ ]:
def preprocess(example):
    example["text"] = example["text"].lower()
    return example

dataset = dataset.map(preprocess)

In [ ]:
dataset = dataset.train_test_split(test_size=0.3)
train_ds = dataset["train"]
test_ds = dataset["test"]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
labels = list(set([x["label"] for x in data]))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

train_ds = train_ds.map(encode_labels)
test_ds = test_ds.map(encode_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return accuracy.compute(predictions=preds, references=labels)

trainer.compute_metrics = compute_metrics
trainer.evaluate()